## Threshold Tuning

This notebook evaluates the impact of the classification threshold on precision and recall for the selected models, using previously tuned hyperparameters. The objective is to identify a threshold that maximizes recall while maintaining an acceptable level of precision.

## Objectives

* Tune the classification threshold of the best-performing model from the previous step.
* Select a threshold that maximizes recall without significantly compromising precision.
* Compare model performance after threshold adjustment and select the final model.

## Configuration

All project configurations are centralized in the `config.yaml` file located in the project root directory.

This file contains parameters related to:

* Data paths
* Model configurations
* Experiment settings

The notebook loads these parameters to ensure reproducibility and consistency across experiments.

## Data Source

This notebook depends on the outputs generated by the following notebooks:

* `3-Feature-Engineering.ipynb`
* `5-hyperparameter_tuning.ipynb`

If these notebooks have not been executed previously, the required datasets and model configurations will not be available.

The training dataset used in this stage is located at:

```
../data/splits/feature_engineered/train_fe.parquet
```

This path is relative to the notebook location within the project structure.

The hyperparameter tuning step provides the best hyperparameters for each model. If it has not been executed, the optimized configurations will not be available.

## Models Evaluated

The following algorithms are evaluated:

* Extra Trees
* XGBoost
* CatBoost

## Research vs Production Code

This notebook was created during the research and experimentation phase of the project.

While it contains exploratory implementations, the production-ready machine learning pipeline is implemented in:

```
src/datapipeline
```

This ensures that the final workflow used in production is modular, testable, and reproducible.


In [ ]:
import pandas as pd
import yaml
import mlflow
import numpy as np
from pathlib import Path
from datapipeline.config.mlflow_config import setup_mlflow
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import tempfile
from sklearn.metrics import (precision_recall_curve, 
                            auc,
                            PrecisionRecallDisplay,
                            precision_score,
                            recall_score)
import matplotlib.pyplot as plt

In [ ]:
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
random_state = config["parameters"]["random_state"]
#artifacts_dir = Path(config['threshold_tuning']['artifacts_path'])

# MLflow

In [ ]:
experiment_name = config['mlflow']['experiment_name']
mlflow_local_folder = config['mlflow']['experiment_path']


In [ ]:
setup_mlflow(experiment_name, mlflow_local_folder)

In [ ]:
client = MlflowClient()

In [ ]:
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id
experiment_id


In [ ]:
mlflow.end_run()
mlflow.start_run(run_name="Threshold_tuning")


# Load  Dataset

In [ ]:
#only the dataset that was previously separated for training is being loaded
dataset_path = Path(config['datasets']['train_feature_engineered_path'])
target_column = config['parameters']['target_column']

In [ ]:
df = pd.read_parquet(dataset_path)

In [ ]:
df_y = df[target_column]

In [ ]:
df_x = df.drop(columns = ['Time', target_column])

In [ ]:
x_train, x_test, y_train, y_test =  train_test_split(df_x, 
                                                     df_y, 
                                                     test_size=0.3, 
                                                     random_state=random_state,
                                                     stratify=df_y)

# Models

Models selected in the previous step:

- Catboost
- Extra Trees
- Xgboost

In [ ]:
#Loading best hyperparameters from the previous notebook

def loading_hyperparameters(experiment_name: str, 
                            random_state: int,
                            tag: str,
                            tag_value: str) -> dict:
    '''
    Get the hyperparameters logged in a mlflow run

    Args:
        experiment_name (str): the name of the mlflow experiment
        random_state (int): the random_stare will be added to the model's hyperparameters
        tag (str): The tag of the run in mlflow
        tag_value (str): the tag value of the run in mlflow
    Returns:
        dict: a dict containing the hyperparameters of a model
        
    '''    
    
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id
    
    experiment_ids = [experiment_id]
    runs = client.search_runs(
        experiment_ids = [experiment_id],
        filter_string = f"tags.{tag} = '{tag_value}'" 
    )
    run_id = runs[0].info.run_id
    run = mlflow.get_run(run_id)
    params = run.data.params
    params['random_state'] = random_state
    return params



In [ ]:
catboost_hyperparameters = loading_hyperparameters(experiment_name=experiment_name,
                                                   random_state=random_state,
                                                   tag='model_name',
                                                   tag_value='catboost')

xgboost_hyperparameters = loading_hyperparameters(experiment_name=experiment_name,
                                                   random_state=random_state,
                                                   tag='model_name',
                                                   tag_value='xgboost')

extratrees_hyperparameters = loading_hyperparameters(experiment_name=experiment_name,
                                                   random_state=random_state,
                                                   tag='model_name',
                                                   tag_value='extratrees')



In [ ]:
catboost_hyperparameters

In [ ]:
xgboost_hyperparameters

In [ ]:
extratrees_hyperparameters

In [ ]:
# The hyperparameters were saved in MLflow as strings and need to be converted back to integers or floats as appropriate.

def to_float(x):
    try:
        if '.' in x:
            return float(x)
        else:
            return int(x)
    except(TypeError, ValueError):
        return x

In [ ]:
#converting the hyperparamenters
catboost_hyperparameters = {key:to_float(value) for key, value in catboost_hyperparameters.items()}
xgboost_hyperparameters = {key:to_float(value) for key, value in xgboost_hyperparameters.items()}
extratrees_hyperparameters = {key:to_float(value) for key, value in extratrees_hyperparameters.items()}

In [ ]:
# instantiating the models
catboost = CatBoostClassifier(**catboost_hyperparameters)
xgboost = xgb.XGBClassifier(**xgboost_hyperparameters)
extratrees = ExtraTreesClassifier(**extratrees_hyperparameters)


In [ ]:
def log_plot_artifacts(plot_object: pd.DataFrame, 
                      file_name: str, 
                      artifacts_path_mlflow: str) -> None:
    
    '''
    Log a plot figure to mlflow 

    Args:
        pllot_object (.plot() object): Object returned by .plot() from matplotlib
        file_name (str): name of the file
        artifacts_path_mlflow (str): folder to save the artifacts in mlflow
    
    '''

    fig = plot_object.figure_

    with tempfile.TemporaryDirectory() as tmp_dir:
        file_path = Path(tmp_dir) / f"{file_name}.png"

        # save plot
        fig.savefig(file_path, bbox_inches="tight")
        
        # log dataset completo
        mlflow.log_artifact(
            str(file_path),
            artifact_path=artifacts_path_mlflow
        )


## Catboost

In [ ]:
# fitting the model
catboost.fit(x_train, y_train, verbose = False)

In [ ]:
y_pred_proba_catboost = catboost.predict_proba(x_test)[:,1]

In [ ]:
precision_cat, recall_cat, thresholds_cat = precision_recall_curve(
    y_test, y_pred_proba_catboost)

In [ ]:
pr_display = PrecisionRecallDisplay(precision=precision_cat, 
                                    recall=recall_cat).plot()
plt.title('CatBoost')
fig = plt.gcf() 
mlflow.log_figure(fig, 'precision_recall_catboost.png' )

In [ ]:
#organizing the results in a pandas dataframe
results_catboost = dict({'Thresholds': thresholds_cat,
                         'Precision': precision_cat[:-1],
                         'Recall': recall_cat[:-1]})
results_catboost = pd.DataFrame(results_catboost)

In [ ]:
results_catboost[(results_catboost['Recall']>0.8) & (results_catboost['Precision']>0.9)]

In [ ]:
def plot_precision_recall_thresold(thresold, precision, recall):
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(thresold, precision[1:], label="Precision")
    ax.plot(thresold, recall[1:], label="Recall")

    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.set_title("Precision and Recall vs Threshold (Catboost)")
    ax.legend()
    ax.grid(True)

    return fig

In [ ]:
fig_catboost = plot_precision_recall_thresold(thresholds_cat, precision_cat, recall_cat)

In [ ]:
mlflow.log_figure(fig_catboost, "precision_recall_vs_threshold_catboost.png")

## XgBoost

In [ ]:
#fitting the model
xgboost.fit(x_train, y_train, verbose = False)

In [ ]:
y_pred_proba_xgboost = xgboost.predict_proba(x_test)[:,1]

In [ ]:
precision_xgb, recall_xgb, thresholds_xgb = precision_recall_curve(
    y_test, y_pred_proba_xgboost)

In [ ]:
#plotting precision vs recall curve
plt.figure()
pr_display = PrecisionRecallDisplay(precision=precision_xgb, 
                                    recall=recall_xgb).plot()
plt.title('Xgboost')
file_name = 'precision_recall_xgboost.png'
fig = plt.gcf() 
mlflow.log_figure(fig, 'precision_recall_xgboost.png' )


In [ ]:
fig_xgboost = plot_precision_recall_thresold(thresholds_xgb, precision_xgb, recall_xgb)

In [ ]:
mlflow.log_figure(fig_xgboost, "precision_recall_vs_threshold_xgboost.png")

In [ ]:
results_xgboost = dict({'Thresholds': thresholds_xgb,
                         'Precision': precision_xgb[:-1],
                         'Recall': recall_xgb[:-1]})
results_xgboost = pd.DataFrame(results_xgboost)

In [ ]:
results_xgboost[(results_xgboost['Recall']>0.8) & (results_xgboost['Precision']>0.9)]

## Extra Trees

In [ ]:
# fitting the model
extratrees.fit(x_train, y_train)

In [ ]:
y_pred_proba_extra = extratrees.predict_proba(x_test)[:,1]

In [ ]:
precision_extra, recall_extra, thresholds_extra = precision_recall_curve(
    y_test, y_pred_proba_extra)

In [ ]:
#plotting precision vs recall curve

plt.figure()
pr_display = PrecisionRecallDisplay(precision=precision_extra, 
                                    recall=recall_extra).plot()
plt.title('Extra Trees')
fig = plt.gcf() 
mlflow.log_figure(fig, 'precision_recall_extratrees.png')
    

In [ ]:
fig_extratrees = plot_precision_recall_thresold(thresholds_extra, precision_extra, recall_extra)

In [ ]:
mlflow.log_figure(fig_extratrees, "precision_recall_vs_threshold_extratrees.png")

In [ ]:
#organizing the results in a pandas dataframe

results_extra = dict({'Thresholds': thresholds_extra,
                         'Precision': precision_extra[:-1],
                         'Recall': recall_extra[:-1]})
results_extra = pd.DataFrame(results_extra)



In [ ]:
results_extra[(results_extra['Recall']>0.8) & (results_extra['Precision']>0.9)]

## Model Comparisson

In [ ]:
# Comparing the results of the three models

In [ ]:
plt.figure()
plt.plot(recall_cat, precision_cat, label = 'CatBoost')
plt.plot(recall_xgb, precision_xgb, label = 'XgBoost')
plt.plot(recall_extra, precision_extra, label = 'XgBoost')

plt.xlabel("Recall")
plt.ylabel('Precision')
plt.legend()
plt.title('CatBoost vs XgBoost vs ExtraTrees')
fig = plt.gcf() 
mlflow.log_figure(fig, 'model_comparison_prcurves.png' )


In [ ]:
area_catboost = auc(recall_cat, precision_cat)
area_xgboost = auc(recall_xgb, precision_xgb)
area_extratrees = auc(recall_extra, precision_extra)

print(f'AUC-PR CatBoost: {area_catboost}')
print(f'AUC-PR XgBoost: {area_xgboost}')
print(f'AUC-PR Extra trees: {area_extratrees}')


In [ ]:
mlflow.log_metrics({'AUC-PR CatBoost:': area_catboost,
                   'AUC-PR XgBoost': area_xgboost,
                   'AUC-PR Extra trees': area_extratrees})

In [ ]:
def generate_dataframe_comparisson(min_recall: float,
                                   df1: pd.DataFrame = results_catboost,
                                   df2: pd.DataFrame = results_xgboost,
                                   df3: pd.DataFrame = results_extra) -> pd.DataFrame: 
    """
    Generates a pandas DataFrame comparing the results of three models
    (CatBoost, XGBoost, and Extra Trees) under a minimum recall constraint.
    
    For each model, the function selects the threshold that achieves the
    lowest recall value that is still greater than or equal to the specified
    minimum recall. It then reports the corresponding threshold, recall,
    and precision.
    
    Args:
        min_recall (float): Minimum required recall.
        df1 (pd.DataFrame): Results for the CatBoost model. The DataFrame must
            contain precision, recall, and threshold values obtained from
            multiple decision thresholds.
        df2 (pd.DataFrame): Results for the XGBoost model. The DataFrame must
            contain precision, recall, and threshold values obtained from
            multiple decision thresholds.
        df3 (pd.DataFrame): Results for the Extra Trees model. The DataFrame must
            contain precision, recall, and threshold values obtained from
            multiple decision thresholds.
    
    Returns:
        pd.DataFrame: A DataFrame containing the results for the three models.
            For each model, the reported values correspond to the threshold
            that achieves the minimum recall above the specified `min_recall`,
            along with the associated precision and recall.
    """

    
    x = df1[df1['Recall']>min_recall]['Recall'].idxmin()
    y = df2[df2['Recall']>min_recall]['Recall'].idxmin()
    z = df3[df3['Recall']>min_recall]['Recall'].idxmin()

    x=df1.loc[x,:]
    y=df2.loc[y,:]
    z=df3.loc[z,:]

    w = pd.concat((x,y,z), axis=1)
    w.columns = ['Catboost', 'XgBoost', 'Extra Trees'] 

    return w



In [ ]:
comparisson = generate_dataframe_comparisson(min_recall=0.83)
comparisson


The CatBoost model shows the best overall performance under the business constraints considered. 
It achieves the highest recall while maintaining precision above 0.8, which is a critical requirement 
for limiting the cost of false positives. Although the Extra Trees model presents the highest PR AUC, 
this result is driven by high recall values obtained at precision levels below the acceptable threshold. 
Since these operating points would generate an excessive number of false positives, they are not suitable 
for deployment in this context.

## Selecting Threshold

In [ ]:
results_catboost['delta Recall(%)'] = 100*(results_catboost["Recall"] - results_catboost["Recall"].shift(-1))/results_catboost["Recall"]

In [ ]:
results_catboost['delta Precision(%)'] = 100*(results_catboost["Precision"] - results_catboost["Precision"].shift(-1))/results_catboost["Precision"]

In [ ]:
results_catboost

In [ ]:
results_catboost[(results_catboost['Recall']>0.8) & (results_catboost['Recall']<0.85)]


The criterion to select the threshold will be primaritly based on the recall metric. That's because in fraud detection, 
failing to identify a fraudulent transaction typically results in direct financial loss and potential reputational damage, 
whereas incorrectly flagging a legitimate transaction usually incurs a lower operational or customer-experience cost. Therefore, 
recall is prioritized over precision.

With an initial threshold of approximately 0.60, the model achieved a recall of 0.81 and a precision of 0.99. When the threshold was reduced to 0.28, recall increased to 0.84, representing a gain of approximately 3.75%. At the same time, precision decreased to 0.95, corresponding to a reduction of about 3.4%.

This trade-off indicates that a relatively small decrease in precision results in a meaningful improvement in recall, increasing the proportion of detected fraud cases while keeping the false-positive rate at an acceptable level.

In [ ]:
mlflow.log_params({'threhold': 0.28})

In [ ]:
mlflow.end_run()